# Foodpanda APAC Customer Feedback Analysis 2024–2026

**Author:** Junior — Data Analyst Portfolio Project

This notebook builds an end-to-end analytical pipeline on ~4.6M Foodpanda customer reviews across 11 APAC markets. The output feeds a Tableau Public dashboard surfacing market-level customer experience patterns, with a secondary keyword analysis identifying what customers consistently praise or complain about per market.

**Business question:** Which APAC markets show signs of customer experience strain, and what specific service dimensions (food, rider, restaurant) are driving sentiment in each?

**Data source:** Public Foodpanda review data scraped by *bwandowando*, distributed via Kaggle. Files cover restaurant metadata and customer reviews for 11 markets, with most countries having 2025 and 2026 versioned snapshots.

**Pipeline outputs:**
- `foodpanda_APAC_final.csv` — analytical master table for Tableau (~4.6M rows, 24 columns)
- `foodpanda_keywords.csv` — keyword frequency and share-of-voice by market, language, and sentiment

**Tooling:** pandas, langdetect, jieba, regex, OneMap API (Singapore geocoding)

**Scope note:** This is a data analyst project, not a data science project. The pipeline focuses on cleaning, joining, classifying, and aggregating. Where NLP-based sentiment was explored, it was ultimately set aside in favour of the explicit star ratings already present in the data — see Section 9 for the reasoning.

## 1. Data Loading

Reads the raw restaurant (`restos`) and review CSV files for 11 APAC markets: Malaysia, Singapore, Philippines, Taiwan, Hong Kong, Bangladesh, Cambodia, Laos, Myanmar, Pakistan, and Thailand.

Most countries have two snapshots (2025 and 2026); these will be reconciled in the deduplication step. Two exceptions are handled separately: Philippines provided unlabelled legacy files alongside the dated ones, and Thailand has only 2025 data following its market exit in May 2025.

The country list was fixed at load time rather than subset later — keeping all 11 markets in the pipeline preserves comparative analysis across the dashboard.

In [1]:
import pandas as pd
import os
import re

In [ ]:
# 11 APAC markets covered by the Kaggle dataset
# Each country loaded into a dict for schema validation before joining

# --- Malaysia ---
my_restos_2025 = pd.read_csv('malaysia/my_restos_2025.csv')
my_restos_2026 = pd.read_csv('malaysia/my_restos_2026.csv')

print("MY Restos 2025:", my_restos_2025.shape)
print("MY Restos 2026:", my_restos_2026.shape)
print("\n2025 Columns:", my_restos_2025.columns.tolist())

In [ ]:
my_reviews_2025 = pd.read_csv('malaysia/my_reviews_2025.csv')
my_reviews_2026 = pd.read_csv('malaysia/my_reviews_2026.csv')

print("MY Reviews 2025:", my_reviews_2025.shape)
print("MY Reviews 2026:", my_reviews_2026.shape)
print("\n2025 Columns:", my_reviews_2025.columns.tolist())

In [ ]:
# --- Singapore ---
print("Singapore files:", os.listdir('singapore/'))

sg_restos_2025 = pd.read_csv('singapore/sg_restos_2025.csv')
sg_restos_2026 = pd.read_csv('singapore/sg_restos_2026.csv')
sg_reviews_2025 = pd.read_csv('singapore/sg_reviews_2025.csv')
sg_reviews_2026 = pd.read_csv('singapore/sg_reviews_2026.csv')

print("\nSG Restos 2025:", sg_restos_2025.shape)
print("SG Restos 2026:", sg_restos_2026.shape)
print("SG Reviews 2025:", sg_reviews_2025.shape)
print("SG Reviews 2026:", sg_reviews_2026.shape)
print("\nSG Restos 2025 Columns:", sg_restos_2025.columns.tolist())

In [ ]:
# --- Philippines ---
# Includes legacy files (restos.csv, reviews.csv) — examined for schema comparison, excluded from analysis
ph_restos_2025 = pd.read_csv('philippines/ph_restos_2025.csv')
ph_reviews_2025 = pd.read_csv('philippines/ph_reviews_2025.csv')
ph_restos_old = pd.read_csv('philippines/restos.csv')
ph_reviews_old = pd.read_csv('philippines/reviews.csv')

print("PH Restos 2025:", ph_restos_2025.shape)
print("PH Reviews 2025:", ph_reviews_2025.shape)
print("PH Restos old:", ph_restos_old.shape)
print("PH Reviews old:", ph_reviews_old.shape)

print("\nPH Restos 2025 columns:", ph_restos_2025.columns.tolist())
print("PH Restos old columns:", ph_restos_old.columns.tolist())

In [ ]:
# --- Taiwan and Hong Kong ---
print("Taiwan files:", os.listdir('taiwan/'))
print("HK files:", os.listdir('hongkong/'))

tw_restos_2025 = pd.read_csv('taiwan/tw_restos_2025.csv')
tw_restos_2026 = pd.read_csv('taiwan/tw_restos_2026.csv')
tw_reviews_2025 = pd.read_csv('taiwan/tw_reviews_2025.csv')
tw_reviews_2026 = pd.read_csv('taiwan/tw_reviews_2026.csv')

hk_restos_2025 = pd.read_csv('hongkong/hk_restos_2025.csv')
hk_restos_2026 = pd.read_csv('hongkong/hk_restos_2026.csv')
hk_reviews_2025 = pd.read_csv('hongkong/hk_reviews_2025.csv')
hk_reviews_2026 = pd.read_csv('hongkong/hk_reviews_2026.csv')

print("\nTW Restos 2025:", tw_restos_2025.shape, "| 2026:", tw_restos_2026.shape)
print("TW Reviews 2025:", tw_reviews_2025.shape, "| 2026:", tw_reviews_2026.shape)
print("HK Restos 2025:", hk_restos_2025.shape, "| 2026:", hk_restos_2026.shape)
print("HK Reviews 2025:", hk_reviews_2025.shape, "| 2026:", hk_reviews_2026.shape)

In [ ]:
# --- Remaining 6 markets ---
market_countries = ['bangladesh', 'cambodia', 'laos', 'myanmar', 'pakistan', 'thailand']

for country in market_countries:
    try:
        files = os.listdir(f'{country}/')
        print(f"{country}: {files}")
    except:
        print(f"{country}: folder not found")

---

## 2. Schema Validation

Before stacking data across countries, schemas are checked for consistency. Two issues surfaced and were resolved here:

1. **Taiwan 2025 reviews** were missing the `restaurant_food` sub-rating column. Added as `None` to preserve column structure during concatenation.
2. **Philippines legacy files** (`restos.csv`, `reviews.csv`) used a different schema and dated to roughly 2020. Excluded from the analysis — combining them would have introduced years of staleness against the 2025–2026 snapshots from every other market.

The validation also confirmed that `overall`, `rider`, and `restaurant_food` are sub-rating columns that decompose the customer's overall star rating into food, rider, and restaurant dimensions. Useful later for diagnosing what's driving sentiment in any given market.

In [ ]:
# Verify schema consistency before any concat — silent column mismatches are a common bug source

# Preview data structure
print("=== MY Restos 2025 Sample ===")
print(my_restos_2025.head(3))

print("\n=== MY Reviews 2025 Sample ===")
print(my_reviews_2025.head(3))

In [ ]:
# Check nulls in both datasets
print("=== MY Restos 2025 Nulls ===")
print(my_restos_2025.isnull().sum())

print("\n=== MY Reviews 2025 Nulls ===")
print(my_reviews_2025.isnull().sum())

In [ ]:
# Taiwan 2025 missing restaurant_food: add as None to preserve column structure
tw_reviews_2025['restaurant_food'] = None

In [ ]:
# Philippines legacy files use a different schema — excluded from analysis
# Add missing Location column to PH 2025 restos to match APAC schema
ph_restos_2025['Location'] = None

> **Limitation:** The Philippines exclusion of legacy files narrows historical coverage for that market. The newer Philippines files (used in this analysis) cover 2025–2026 only.

---

## 3. Deduplication Across 2025 and 2026 Snapshots

Most countries provided both 2025 and 2026 snapshot files. Where a `StoreId` (restaurants) or `uuid` (reviews) appeared in both years, the 2026 record is retained as the more current version.

Approach: tag each file with a `source_year`, concatenate, sort descending by year, then drop duplicates on the appropriate key. This guarantees the 2026 record wins on overlap.

Overlap volumes were measured before deduplication and logged for transparency. For example, Malaysia had ~25k restaurant overlaps and ~85k review overlaps between the two snapshots — substantial enough to materially affect downstream counts if not handled.

In [ ]:
# --- Malaysia deduplication ---

# Check for duplicate reviews using uuid (should be unique per review)
print("MY Reviews 2025 duplicate uuids:", my_reviews_2025['uuid'].duplicated().sum())
print("MY Reviews 2026 duplicate uuids:", my_reviews_2026['uuid'].duplicated().sum())

# Overlap counts logged before dedup as a sanity check
overlap = my_reviews_2025['uuid'].isin(my_reviews_2026['uuid']).sum()
print("\nReviews appearing in BOTH 2025 and 2026 files:", overlap)

In [ ]:
# Sort descending by source_year so 2026 is kept first; drop_duplicates keeps first by default

# Dedup reviews
my_reviews_2025['source_year'] = 2025
my_reviews_2026['source_year'] = 2026

my_reviews_combined = pd.concat([my_reviews_2025, my_reviews_2026], ignore_index=True)
my_reviews_combined = my_reviews_combined.sort_values('source_year', ascending=False)
my_reviews_combined = my_reviews_combined.drop_duplicates(subset='uuid', keep='first')

print("2025 reviews:", len(my_reviews_2025))
print("2026 reviews:", len(my_reviews_2026))
print("Combined (deduped):", len(my_reviews_combined))

In [ ]:
# Dedup restos
resto_overlap = my_restos_2025['StoreId'].isin(my_restos_2026['StoreId']).sum()
print("Restos appearing in BOTH 2025 and 2026:", resto_overlap)

my_restos_2025['source_year'] = 2025
my_restos_2026['source_year'] = 2026

my_restos_combined = pd.concat([my_restos_2025, my_restos_2026], ignore_index=True)
my_restos_combined = my_restos_combined.sort_values('source_year', ascending=False)
my_restos_combined = my_restos_combined.drop_duplicates(subset='StoreId', keep='first')

print("MY Restos unique:", len(my_restos_combined))

In [ ]:
# --- Singapore deduplication ---

sg_resto_overlap = sg_restos_2025['StoreId'].isin(sg_restos_2026['StoreId']).sum()
sg_review_overlap = sg_reviews_2025['uuid'].isin(sg_reviews_2026['uuid']).sum()

print("SG Resto overlap:", sg_resto_overlap)
print("SG Review overlap:", sg_review_overlap)

sg_restos_2025['source_year'] = 2025
sg_restos_2026['source_year'] = 2026
sg_reviews_2025['source_year'] = 2025
sg_reviews_2026['source_year'] = 2026

sg_restos_combined = pd.concat([sg_restos_2025, sg_restos_2026], ignore_index=True)
sg_restos_combined = sg_restos_combined.sort_values('source_year', ascending=False)
sg_restos_combined = sg_restos_combined.drop_duplicates(subset='StoreId', keep='first')

sg_reviews_combined = pd.concat([sg_reviews_2025, sg_reviews_2026], ignore_index=True)
sg_reviews_combined = sg_reviews_combined.sort_values('source_year', ascending=False)
sg_reviews_combined = sg_reviews_combined.drop_duplicates(subset='uuid', keep='first')

print("\nSG Restos unique:", len(sg_restos_combined))
print("SG Reviews unique:", len(sg_reviews_combined))

In [ ]:
# --- Taiwan and Hong Kong deduplication ---

tw_restos_2025['source_year'] = 2025
tw_restos_2026['source_year'] = 2026
tw_reviews_2025['source_year'] = 2025
tw_reviews_2026['source_year'] = 2026

hk_restos_2025['source_year'] = 2025
hk_restos_2026['source_year'] = 2026
hk_reviews_2025['source_year'] = 2025
hk_reviews_2026['source_year'] = 2026

tw_resto_overlap = tw_restos_2025['StoreId'].isin(tw_restos_2026['StoreId']).sum()
tw_review_overlap = tw_reviews_2025['uuid'].isin(tw_reviews_2026['uuid']).sum()
hk_resto_overlap = hk_restos_2025['StoreId'].isin(hk_restos_2026['StoreId']).sum()
hk_review_overlap = hk_reviews_2025['uuid'].isin(hk_reviews_2026['uuid']).sum()

print("TW Resto overlap:", tw_resto_overlap)
print("TW Review overlap:", tw_review_overlap)
print("HK Resto overlap:", hk_resto_overlap)
print("HK Review overlap:", hk_review_overlap)

# Taiwan
tw_restos_combined = pd.concat([tw_restos_2025, tw_restos_2026], ignore_index=True)
tw_restos_combined = tw_restos_combined.sort_values('source_year', ascending=False)
tw_restos_combined = tw_restos_combined.drop_duplicates(subset='StoreId', keep='first')

tw_reviews_combined = pd.concat([tw_reviews_2025, tw_reviews_2026], ignore_index=True)
tw_reviews_combined = tw_reviews_combined.sort_values('source_year', ascending=False)
tw_reviews_combined = tw_reviews_combined.drop_duplicates(subset='uuid', keep='first')

# Hong Kong
hk_restos_combined = pd.concat([hk_restos_2025, hk_restos_2026], ignore_index=True)
hk_restos_combined = hk_restos_combined.sort_values('source_year', ascending=False)
hk_restos_combined = hk_restos_combined.drop_duplicates(subset='StoreId', keep='first')

hk_reviews_combined = pd.concat([hk_reviews_2025, hk_reviews_2026], ignore_index=True)
hk_reviews_combined = hk_reviews_combined.sort_values('source_year', ascending=False)
hk_reviews_combined = hk_reviews_combined.drop_duplicates(subset='uuid', keep='first')

print("\nTW Restos unique:", len(tw_restos_combined), "| Reviews:", len(tw_reviews_combined))
print("HK Restos unique:", len(hk_restos_combined), "| Reviews:", len(hk_reviews_combined))

---

## 4. Joining Restaurants to Reviews

Reviews are joined to restaurant metadata using `StoreId` as the join key, with a LEFT JOIN to preserve every review even where restaurant metadata is missing. A `country` column is added at this stage so the downstream stacked table is country-aware.

Note: joining two dataframes that both contain a `source_year` column produces `source_year_x` and `source_year_y` artifacts. These are dropped — they served their purpose during deduplication and are no longer informative.

In [ ]:
# --- Malaysia ---
# LEFT JOIN: keep every review, even if restaurant metadata is missing
master_MY = my_reviews_combined.merge(my_restos_combined, on='StoreId', how='left')
master_MY['country'] = 'Malaysia'

# Drop _x/_y suffix columns — artifact of joining two frames with the same column name
master_MY = master_MY.drop(columns=['source_year_x', 'source_year_y'], errors='ignore')

print("Master Malaysia shape:", master_MY.shape)
print("\nNull check on key columns:")
print(master_MY[['CompleteStoreName', 'City', 'Location', 'text']].isnull().sum())

In [ ]:
# --- Singapore ---
master_SG = sg_reviews_combined.merge(sg_restos_combined, on='StoreId', how='left')
master_SG['country'] = 'Singapore'

print("Master Singapore shape:", master_SG.shape)
print("\nNull check:")
print(master_SG[['CompleteStoreName', 'City', 'Location', 'text']].isnull().sum())

In [ ]:
# --- Philippines ---
# Only 2025 files used — legacy files excluded (see Section 2)
master_PH = ph_reviews_2025.merge(ph_restos_2025, on='StoreId', how='left')
master_PH['country'] = 'Philippines'

print("Master Philippines shape:", master_PH.shape)
print("\nNull check:")
print(master_PH[['CompleteStoreName', 'City', 'text']].isnull().sum())

In [ ]:
# --- Taiwan ---
master_TW = tw_reviews_combined.merge(tw_restos_combined, on='StoreId', how='left')
master_TW['country'] = 'Taiwan'
master_TW = master_TW.drop(columns=['source_year_x', 'source_year_y'], errors='ignore')

# --- Hong Kong ---
master_HK = hk_reviews_combined.merge(hk_restos_combined, on='StoreId', how='left')
master_HK['country'] = 'Hong Kong'
master_HK = master_HK.drop(columns=['source_year_x', 'source_year_y'], errors='ignore')

print("Master Taiwan shape:", master_TW.shape)
print("Master Hong Kong shape:", master_HK.shape)
print("\nTW Null check:")
print(master_TW[['CompleteStoreName', 'City', 'text']].isnull().sum())
print("\nHK Null check:")
print(master_HK[['CompleteStoreName', 'City', 'text']].isnull().sum())

In [ ]:
# --- Remaining 6 markets ---
# Generic function to load, deduplicate, and join for any country

def load_and_process_country(country_folder, prefix, country_name):
    """Load, deduplicate and join restos + reviews for any country."""
    
    files = os.listdir(f'{country_folder}/')
    resto_files = sorted([f for f in files if 'restos' in f])
    review_files = sorted([f for f in files if 'reviews' in f])
    
    print(f"\n=== {country_name} ===")
    
    # Load restos
    resto_dfs = []
    for f in resto_files:
        df = pd.read_csv(f'{country_folder}/{f}', low_memory=False)
        year = int(''.join(filter(str.isdigit, f.split('restos_')[-1].split('.')[0])))
        df['source_year'] = year
        resto_dfs.append(df)
        print(f"Loaded {f}: {df.shape}")
    
    restos_combined = pd.concat(resto_dfs, ignore_index=True)
    restos_combined = restos_combined.sort_values('source_year', ascending=False)
    restos_combined = restos_combined.drop_duplicates(subset='StoreId', keep='first')
    restos_combined = restos_combined.drop(columns=['source_year'], errors='ignore')
    
    # Load reviews
    review_dfs = []
    for f in review_files:
        df = pd.read_csv(f'{country_folder}/{f}', low_memory=False)
        year = int(''.join(filter(str.isdigit, f.split('reviews_')[-1].split('.')[0])))
        df['source_year'] = year
        review_dfs.append(df)
        print(f"Loaded {f}: {df.shape}")
    
    reviews_combined = pd.concat(review_dfs, ignore_index=True)
    reviews_combined = reviews_combined.sort_values('source_year', ascending=False)
    reviews_combined = reviews_combined.drop_duplicates(subset='uuid', keep='first')
    reviews_combined = reviews_combined.drop(columns=['source_year'], errors='ignore')
    
    # Join
    master = reviews_combined.merge(restos_combined, on='StoreId', how='left')
    master['country'] = country_name
    master = master.drop(columns=['source_year_x', 'source_year_y'], errors='ignore')
    
    print(f"Master {country_name} shape: {master.shape}")
    print(f"Null check - text: {master['text'].isnull().sum()}, CompleteStoreName: {master['CompleteStoreName'].isnull().sum()}")
    
    return master

master_BD = load_and_process_country('bangladesh', 'bd', 'Bangladesh')
master_KH = load_and_process_country('cambodia', 'kh', 'Cambodia')
master_LA = load_and_process_country('laos', 'la', 'Laos')
master_MM = load_and_process_country('myanmar', 'mm', 'Myanmar')
master_PK = load_and_process_country('pakistan', 'pk', 'Pakistan')
master_TH = load_and_process_country('thailand', 'th', 'Thailand')

---

## 5. Stacking All Countries Into Master Table

All 11 country dataframes are concatenated into a single master table (`master_APAC`) of approximately 4.6M rows.

A checkpoint CSV is saved at this stage so subsequent steps can resume without reloading and rejoining the source files — significant given the master file is ~1.5GB.

In [ ]:
# Executed once during pipeline build; cached output loaded below for inspection
# Checkpoint saved here — re-running downstream cells does not require re-running Sections 1–4

# --- Original stacking code ---
# master_APAC = pd.concat([
#     master_MY,
#     master_SG,
#     master_PH,
#     master_TW,
#     master_HK,
#     master_BD,
#     master_KH,
#     master_LA,
#     master_MM,
#     master_PK,
#     master_TH
# ], ignore_index=True)
#
# master_APAC.to_csv('master_APAC_checkpoint.csv', index=False)
# print("Checkpoint saved successfully")

In [2]:
# Load checkpoint for downstream sections
master_APAC = pd.read_csv('master_APAC_checkpoint.csv', low_memory=False)

print("master_APAC loaded:", master_APAC.shape)
print("\nCountry breakdown:")
print(master_APAC['country'].value_counts())

master_APAC loaded: (4604685, 26)

Country breakdown:
country
Pakistan       1552208
Malaysia        906603
Taiwan          835107
Bangladesh      492140
Philippines     213192
Thailand        190024
Singapore       147019
Myanmar          83034
Hong Kong        71689
Cambodia         69919
Laos             43750
Name: count, dtype: int64


In [3]:
# Load the final pipeline output as the working dataframe for all downstream sections.
# foodpanda_APAC_final.csv contains the complete output of Sections 1–11:
# stacked data, geocoded regions, engineered features, language tags.
final_df = pd.read_csv('foodpanda_APAC_final.csv', low_memory=False)
print(f"final_df loaded: {final_df.shape}")
print(f"Columns: {list(final_df.columns)}")

# Secondary output for Section 10 (keyword analysis)
keywords_df = pd.read_csv('foodpanda_keywords.csv')
print(f"\nkeywords_df loaded: {keywords_df.shape}")

final_df loaded: (4604685, 24)
Columns: ['StoreId', 'uuid', 'country', 'CompleteStoreName', 'FoodType', 'AverageRating', 'Reviewers', 'City', 'Location', 'region', 'latitude', 'longitude', 'createdAt', 'text', 'overall', 'rider', 'restaurant_food', 'likeCount', 'isAnonymous', 'rating_sentiment', 'sentiment_label', 'year', 'month', 'year_month']

keywords_df loaded: (480, 7)


---

## 6. Singapore Region Assignment via Geocoding

Assigns Singapore restaurants to one of the five Singapore planning regions (Central, North, North-East, East, West) so the dashboard can surface region-level performance for the Singapore market.

Three approaches were attempted in sequence:

1. **Keyword mapping on the `Location` field.** Built a dictionary of neighbourhood keywords and applied regex matching. Result: ~20% unknown — too high to ship.
2. **Postal code extraction.** Singapore's first two postal digits map deterministically to planning districts. Regex-extracted six-digit codes from address strings. Result: 53% coverage.
3. **OneMap API (Singapore Land Authority).** For the 3,225 unique restaurants without parseable postal codes, addresses were sent to the OneMap geocoding API. Only 398 returned a valid match (87% failure rate), driven by inconsistent and incomplete address formatting in the source data. Final coverage reached ~60%.

The remaining 40% are flagged as `Unknown` rather than imputed. Imputing region without evidence would distort regional comparisons in the dashboard.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Methodology reference — not executed in this notebook.
# Output is preserved in foodpanda_APAC_final.csv (loaded above as final_df).
# ─────────────────────────────────────────────────────────────────

# --- Approach 2: Postal code extraction ---

def extract_postal_code(location):
    if pd.isna(location):
        return None
    
    location_clean = location.strip()
    
    # Pattern 1: Standard 6 digits with word boundary e.g. 409051
    match = re.search(r'\b(\d{6})\b', location_clean)
    if match:
        return match.group(1)
    
    # Pattern 2: S + 6 digits e.g. S409051, S(409051)
    match = re.search(r'[Ss]\s*\(?(\d{6})\)?', location_clean)
    if match:
        return match.group(1)
    
    # Pattern 3: SG + 6 digits e.g. SG530105
    match = re.search(r'[Ss][Gg]\s*(\d{6})', location_clean)
    if match:
        return match.group(1)
    
    return None

# Apply to Singapore subset
sg_mask = master_APAC['country'] == 'Singapore'
master_APAC.loc[sg_mask, 'postal_code'] = master_APAC.loc[sg_mask, 'Location'].apply(extract_postal_code)

total_sg = sg_mask.sum()
found = master_APAC.loc[sg_mask, 'postal_code'].notna().sum()
print(f"Postal codes found: {found} ({round(found/total_sg*100, 2)}%)")
print(f"Postal codes missing: {total_sg - found} ({round((total_sg - found)/total_sg*100, 2)}%)")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Methodology reference — not executed in this notebook.
# Output is preserved in foodpanda_APAC_final.csv (loaded above as final_df).
# ─────────────────────────────────────────────────────────────────

# Singapore postal code district to region mapping
# Based on first 2 digits of postal code
district_to_region = {
    '01': 'Central', '02': 'Central', '03': 'Central', '04': 'Central',
    '05': 'Central', '06': 'Central', '07': 'Central', '08': 'Central',
    '09': 'Central', '10': 'Central', '11': 'Central', '12': 'Central',
    '13': 'Central', '14': 'Central', '15': 'Central', '16': 'Central',
    '17': 'Central', '18': 'Central', '19': 'Central', '20': 'Central',
    '21': 'Central', '22': 'West',    '23': 'North',   '24': 'North',
    '25': 'North',   '26': 'North',   '27': 'North',   '28': 'North-East',
    '29': 'North-East', '30': 'North-East', '31': 'Central', '32': 'Central',
    '33': 'Central', '34': 'North-East', '35': 'North-East', '36': 'North-East',
    '37': 'North-East', '38': 'North-East', '39': 'North-East', '40': 'North-East',
    '41': 'North-East', '42': 'East', '43': 'East', '44': 'East',
    '45': 'East',    '46': 'East',    '47': 'East',    '48': 'East',
    '49': 'East',    '50': 'East',    '51': 'East',    '52': 'East',
    '53': 'North-East', '54': 'North-East', '55': 'North-East', '56': 'North',
    '57': 'North',   '58': 'Central', '59': 'West',    '60': 'West',
    '61': 'West',    '62': 'West',    '63': 'West',    '64': 'West',
    '65': 'West',    '66': 'West',    '67': 'West',    '68': 'West',
    '69': 'West',    '70': 'West',    '71': 'West',    '72': 'East',
    '73': 'East',    '75': 'East',    '76': 'East',    '77': 'East',
    '78': 'East',    '79': 'East',    '80': 'East',    '81': 'East',
    '82': 'North-East'
}

def get_region_from_postal(postal_code):
    if pd.isna(postal_code) or len(str(postal_code)) < 2:
        return 'Unknown'
    prefix = str(postal_code)[:2]
    return district_to_region.get(prefix, 'Unknown')

# Apply region mapping
master_APAC.loc[sg_mask, 'region'] = master_APAC.loc[sg_mask, 'postal_code'].apply(get_region_from_postal)

print("Region distribution (postal code only):")
print(master_APAC.loc[sg_mask, 'region'].value_counts())
print("\nUnknown %:", round(master_APAC.loc[sg_mask, 'region'].eq('Unknown').mean() * 100, 2), "%")

### Approach 3: OneMap API geocoding

The following code was executed once to geocode the 3,225 unique restaurants missing parseable postal codes. Results were saved to `singapore/geocode_checkpoint.csv` and are loaded below.

In [ ]:
# Executed once during pipeline build; cached output loaded below for inspection
# OneMap API token loaded from environment — never hardcode or commit credentials

# import requests
# import time
#
# ONEMAP_TOKEN = os.environ.get('ONEMAP_TOKEN')
#
# def geocode_address(search_val, token):
#     if pd.isna(search_val) or str(search_val).strip() == '':
#         return None, None, None
#     
#     url = "https://www.onemap.gov.sg/api/common/elastic/search"
#     params = {
#         "searchVal": str(search_val).strip(),
#         "returnGeom": "Y",
#         "getAddrDetails": "Y",
#         "pageNum": 1
#     }
#     headers = {"Authorization": token}
#     
#     try:
#         response = requests.get(url, params=params, headers=headers, timeout=10)
#         data = response.json()
#         if data.get("found", 0) > 0:
#             result = data["results"][0]
#             return (
#                 float(result["LATITUDE"]),
#                 float(result["LONGITUDE"]),
#                 result.get("POSTAL", None)
#             )
#     except Exception as e:
#         return None, None, None
#     
#     return None, None, None
#
# # Geocoding loop with checkpoint saving every 500 rows
# missing_postal_stores = master_APAC[
#     (master_APAC['country'] == 'Singapore') &
#     (master_APAC['postal_code'].isna())
# ][['StoreId', 'Location']].drop_duplicates(subset='StoreId')
#
# results = []
# for i, (_, row) in enumerate(missing_postal_stores.iterrows()):
#     lat, lng, postal = geocode_address(row['Location'], ONEMAP_TOKEN)
#     results.append({
#         'StoreId': row['StoreId'],
#         'latitude': lat,
#         'longitude': lng,
#         'postal_recovered': postal
#     })
#     if (i + 1) % 500 == 0:
#         pd.DataFrame(results).to_csv('singapore/geocode_checkpoint.csv', index=False)
#     time.sleep(0.3)  # Respectful rate limiting
#
# geocoded_df = pd.DataFrame(results)
# geocoded_df.to_csv('singapore/geocode_checkpoint.csv', index=False)
# print(f"Successfully geocoded: {geocoded_df['latitude'].notna().sum()}")
# print(f"Failed: {geocoded_df['latitude'].isna().sum()}")

In [ ]:
# --- Exploratory: OneMap debug call (set aside) ---
# Used during development to inspect what fields OneMap returns
#
# def geocode_onemap_debug(search_val, token):
#     url = "https://www.onemap.gov.sg/api/common/elastic/search"
#     params = {
#         "searchVal": search_val,
#         "returnGeom": "Y",
#         "getAddrDetails": "Y",
#         "pageNum": 1
#     }
#     headers = {"Authorization": token}
#     response = requests.get(url, params=params, headers=headers)
#     data = response.json()
#     if data.get("found", 0) > 0:
#         print("All available fields:")
#         for key, value in data["results"][0].items():
#             print(f"  {key}: {value}")

In [4]:
# Verification: Singapore region distribution in final output
sg_final = final_df[final_df['country'] == 'Singapore']
print("Singapore region distribution (from foodpanda_APAC_final.csv):")
print(sg_final['region'].value_counts())
print(f"\nCoverage (non-Unknown): {(sg_final['region'] != 'Unknown').mean() * 100:.1f}%")
print(f"Geocoded records (lat/long populated): {sg_final['latitude'].notna().sum()}")

Singapore region distribution (from foodpanda_APAC_final.csv):
region
Unknown       59217
East          28033
Central       19632
North-East    19382
West          14641
North          6114
Name: count, dtype: int64

Coverage (non-Unknown): 59.7%
Geocoded records (lat/long populated): 9438


> **Limitation (methodology caveat):** Singapore region coverage is approximately 60%. The remaining 40% are labelled "Unknown" and shown as a distinct category in the dashboard rather than excluded or imputed. Regional comparisons within Singapore should be read with this in mind.
>
> Latitude and longitude were captured for the 398 successfully geocoded records but are not surfaced in the dashboard — coverage is too sparse for a meaningful map. A future iteration would standardise addresses before geocoding to lift the OneMap success rate.

---

## 7. Feature Engineering

Adds the analytical fields used by the Tableau dashboard:

- **`rating_sentiment`** — maps the 1–5 star rating to a continuous −1 to +1 scale, mirroring the conventional NLP sentiment range. This makes the field intuitive to aggregate (averages preserve direction) and comparable to outputs from any future NLP layer.
- **`sentiment_label`** — a categorical version (Positive / Neutral / Negative) for grouped chart use.
- **`year`, `month`, `year_month`** — time partitions extracted from `createdAt`. `year_month` is used as the trend axis in the dashboard.
- **`createdAt`** is parsed to UTC datetime for consistent time handling across markets.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Methodology reference — not executed in this notebook.
# Output is preserved in foodpanda_APAC_final.csv (loaded above as final_df).
# ─────────────────────────────────────────────────────────────────

# rating_sentiment matches the conventional NLP scale (-1 to +1) for comparability
def rating_to_sentiment(rating):
    if pd.isna(rating):
        return None
    mapping = {
        1.0: -1.0,
        2.0: -0.5,
        3.0:  0.0,
        4.0:  0.5,
        5.0:  1.0
    }
    return mapping.get(float(rating), None)

master_APAC['rating_sentiment'] = master_APAC['overall'].apply(rating_to_sentiment)

# Sentiment label
def sentiment_label(score):
    if pd.isna(score):
        return 'Unknown'
    elif score >= 0.5:
        return 'Positive'
    elif score <= -0.5:
        return 'Negative'
    else:
        return 'Neutral'

master_APAC['sentiment_label'] = master_APAC['rating_sentiment'].apply(sentiment_label)

# year_month as a string period is Tableau-friendly and avoids timezone quirks in the trend chart
master_APAC['createdAt'] = pd.to_datetime(master_APAC['createdAt'], utc=True, errors='coerce')
master_APAC['year'] = master_APAC['createdAt'].dt.year
master_APAC['month'] = master_APAC['createdAt'].dt.month
master_APAC['year_month'] = master_APAC['createdAt'].dt.to_period('M').astype(str)

print("New columns added successfully")
print("\nRating sentiment distribution:")
print(master_APAC['sentiment_label'].value_counts())
print("\nYear distribution:")
print(master_APAC['year'].value_counts().sort_index())
print("\nSample:")
print(master_APAC[['country', 'overall', 'rating_sentiment', 
                    'sentiment_label', 'year', 'month', 'year_month']].head(10))

In [5]:
# Verification: engineered features exist in final output
print("rating_sentiment distribution:")
print(final_df['rating_sentiment'].value_counts().sort_index())
print("\nsentiment_label distribution:")
print(final_df['sentiment_label'].value_counts())
print("\nyear_month range:", final_df['year_month'].min(), "to", final_df['year_month'].max())

rating_sentiment distribution:
rating_sentiment
-1.0    1214395
-0.5     350996
 0.0     491367
 0.5     484337
 1.0    2063590
Name: count, dtype: int64

sentiment_label distribution:
sentiment_label
Positive    2547927
Negative    1565391
Neutral      491367
Name: count, dtype: int64

year_month range: 2024-02 to 2026-01


---

## 8. Language Detection

Language is detected only for the five markets where multilingual sentiment matters for the keyword analysis: Malaysia, Singapore, Philippines, Taiwan, and Hong Kong. The remaining six markets are tagged `language = 'skip'` — they are out of scope for the keyword analysis (see Section 10), so detection cost is unnecessary.

**A finding worth surfacing:** `langdetect` initially classified ~538k reviews as Korean (`ko`). Sampling these confirmed they were almost all Traditional Chinese — a known weakness of `langdetect` with short Traditional Chinese text. All `ko` labels were remapped to `zh-tw`. Without this correction, Taiwan's keyword analysis would have been almost entirely missing.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Methodology reference — not executed in this notebook.
# Language detection was an intermediate step for keyword analysis (Section 10).
# The 'language' column is not included in the final export.
# ─────────────────────────────────────────────────────────────────

# Executed once during pipeline build; cached output loaded below for inspection
# Language detection on ~2M+ rows takes approximately 30 minutes

# from langdetect import detect, LangDetectException
#
# def detect_language(text):
#     if pd.isna(text) or str(text).strip() == '':
#         return 'unknown'
#     try:
#         return detect(str(text))
#     except LangDetectException:
#         return 'unknown'
#
# # 'skip' tag for non-sentiment countries — keyword analysis only runs on the 5 sentiment markets
# sentiment_countries = ['Malaysia', 'Singapore', 'Philippines', 'Taiwan', 'Hong Kong']
# master_APAC['language'] = 'skip'
#
# mask = master_APAC['country'].isin(sentiment_countries)
# print(f"Rows to detect language: {mask.sum():,}")
# print(f"Rows skipped (non-sentiment countries): {(~mask).sum():,}")
#
# master_APAC.loc[mask, 'language'] = master_APAC.loc[mask, 'text'].apply(detect_language)
# master_APAC.to_csv('master_APAC_checkpoint.csv', index=False)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Methodology reference — not executed in this notebook.
# Language detection was an intermediate step for keyword analysis (Section 10).
# The 'language' column is not included in the final export.
# ─────────────────────────────────────────────────────────────────

# Verify language detection results from checkpoint
sentiment_countries = ['Malaysia', 'Singapore', 'Philippines', 'Taiwan', 'Hong Kong']
mask = master_APAC['country'].isin(sentiment_countries)

print("Language distribution (sentiment countries only):")
print(master_APAC.loc[mask, 'language'].value_counts().head(20))

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Methodology reference — not executed in this notebook.
# Language detection was an intermediate step for keyword analysis (Section 10).
# The 'language' column is not included in the final export.
# ─────────────────────────────────────────────────────────────────

# 'ko' → 'zh-tw' remap: langdetect misclassifies short Traditional Chinese as Korean
# Investigate Korean detections — sample confirms these are Traditional Chinese
korean_count = (master_APAC['language'] == 'ko').sum()
print(f"Reviews tagged as Korean: {korean_count}")

if korean_count > 0:
    korean_samples = master_APAC[master_APAC['language'] == 'ko']['text'].dropna().sample(
        min(20, korean_count), random_state=42
    )
    print("\nSample 'Korean' reviews (actually Traditional Chinese):")
    for i, text in enumerate(korean_samples):
        print(f"{i+1}. '{text}'")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Methodology reference — not executed in this notebook.
# Language detection was an intermediate step for keyword analysis (Section 10).
# The 'language' column is not included in the final export.
# ─────────────────────────────────────────────────────────────────

# Remap misclassified Korean → Traditional Chinese
master_APAC['language'] = master_APAC['language'].replace('ko', 'zh-tw')

print("Updated language distribution:")
print(master_APAC[master_APAC['country'].isin(sentiment_countries)]['language'].value_counts().head(15))

In [6]:
# Verification: language detection impact on final output
# The 'language' column was an intermediate feature used for keyword analysis (Section 10).
# It is not included in the 24-column final export.
sentiment_countries = ['Malaysia', 'Singapore', 'Philippines', 'Taiwan', 'Hong Kong']
if 'language' in final_df.columns:
    lang_df = final_df[final_df['country'].isin(sentiment_countries)]
    print("Language distribution (sentiment countries):")
    print(lang_df['language'].value_counts().head(10))
else:
    print("Note: 'language' column not in final export.")
    print("Detection output was used upstream for keyword analysis only (Section 10).")
    print(f"\nKeyword analysis covered {keywords_df['country'].nunique()} countries:")
    print(keywords_df['country'].unique())

Note: 'language' column not in final export.
Detection output was used upstream for keyword analysis only (Section 10).

Keyword analysis covered 5 countries:
['Malaysia' 'Singapore' 'Philippines' 'Taiwan' 'Hong Kong']


> **Limitation:** `langdetect` is statistical and prone to misclassification on short text. The Korean → Traditional Chinese correction was discoverable through sampling, but smaller misclassifications likely remain. Language tags should be treated as approximate, not authoritative.

---

## 9. NLP Sentiment Scoring (Exploratory)

Multilingual NLP sentiment was explored before the final decision to rely on star ratings instead. This section documents what was tried and why each path was set aside — kept in the notebook for transparency rather than removed.

**VADER (English).** Installed and tested successfully. Scored a sample positive review at 0.85, in line with expectations.

**SnowNLP (Chinese).** Installed and tested successfully. Scored a sample positive Chinese review at 0.976, normalised to the −1 to +1 range.

**Malaya (Malay NLP).** Installation succeeded but import failed under Python 3.14 due to a `torch` / `transformers` dependency conflict. A fix would require either a Python 3.10/3.11 environment or waiting for Malaya to update its dependency pins. Set aside for this iteration.

**Multilingual transformer (`tabularisai`).** Tested on Malay, Tagalog, and Rojak (mixed Malay-English) reviews. Accurate on clear positives and negatives, but failed on the mixed-sentiment constructions common in Rojak. A representative failure: *"Lambat gila delivery, dah sejuk semua"* ("Delivery was insanely slow, everything's gone cold") was scored as Positive. Accuracy was insufficient for the volume of Malaysian reviews this would touch.

### Decision: use star ratings as the sentiment indicator throughout the pipeline.

The dataset contains 4.6M explicit star ratings — directly provided by customers, language-agnostic, and not subject to model error. NLP sentiment is most valuable when explicit ratings are absent; here, modelling sentiment when the customer has already supplied it would have introduced noise without improving signal.

This is also the reason the project is named *Customer Feedback Analysis* rather than *Sentiment Analysis* — the analytical method should be reflected in the title.

In [ ]:
# Exploratory cells — kept for transparency. Final pipeline uses star ratings (see Section 7)

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from snownlp import SnowNLP

vader = SentimentIntensityAnalyzer()

# VADER test (English)
test_english = "The food was absolutely amazing, delivery was super fast!"
vader_score = vader.polarity_scores(test_english)['compound']
print(f"VADER (English): '{test_english}'")
print(f"Score: {vader_score}\n")

# SnowNLP test (Chinese)
test_chinese = "食物非常好吃，送货速度很快！"
snow_score = SnowNLP(test_chinese).sentiments
snow_normalised = (snow_score * 2) - 1  # normalise 0-1 to -1 to +1
print(f"SnowNLP (Chinese): '{test_chinese}'")
print(f"Score: {snow_normalised}")

In [ ]:
# --- Attempted: apply VADER/SnowNLP to full dataset (set aside) ---
# This was executed but the resulting sentiment_score column was ultimately
# dropped in favour of rating_sentiment (Section 7).

# def get_sentiment(text, language):
#     if pd.isna(text) or str(text).strip() == '':
#         return None
#     try:
#         if language == 'en':
#             score = vader.polarity_scores(str(text))['compound']
#             return score
#         elif language in ['zh-tw', 'zh-cn']:
#             score = SnowNLP(str(text)).sentiments
#             return (score * 2) - 1
#         else:
#             return None
#     except Exception as e:
#         return None
#
# sentiment_mask = master_APAC['country'].isin(sentiment_countries)
# master_APAC['sentiment_score'] = None
# master_APAC.loc[sentiment_mask, 'sentiment_score'] = master_APAC.loc[sentiment_mask].apply(
#     lambda row: get_sentiment(row['text'], row['language']), axis=1
# )

In [ ]:
# --- Attempted: Malaya (Malay NLP) — set aside ---
# Malaya import failure: Python 3.14 incompatibility with transformer dependencies
#
# import malaya
# malaya_sentiment = malaya.sentiment.huggingface()
#
# test_reviews = [
#     "Makanan sangat sedap dan penghantaran cepat!",
#     "Makanan sejuk bila sampai, sangat kecewa",
#     "Ok je, biasa biasa",
#     "Memang best! Akan order lagi",
#     "Lambat sangat, dah lapar tunggu lama",
# ]
#
# Installation succeeded but import failed with:
#   ImportError: torch/transformers version conflict under Python 3.14
# Would require Python 3.10/3.11 environment to resolve.

In [ ]:
# --- Attempted: tabularisai multilingual transformer — set aside ---
# tabularisai failure mode: mixed-sentiment Rojak sentences misclassified

from transformers import pipeline
import torch

print("Loading multilingual sentiment model...")
sentiment_pipeline = pipeline(
    "text-classification",
    model="tabularisai/multilingual-sentiment-analysis",
    device=-1  # CPU
)

test_reviews = [
    # English
    "The food was amazing and delivery was super fast!",
    "Cold food, very disappointed with the service",
    # Malay
    "Makanan sangat sedap dan penghantaran cepat!",
    "Makanan sejuk bila sampai, sangat kecewa",
    # Rojak (mixed Malay-English)
    "Sedap gila, memang best! Will order again lah",
    "Lambat gila delivery, dah sejuk semua",
    # Chinese
    "食物非常好吃，送货速度很快！",
    "食物冷掉了，非常失望",
    # Tagalog
    "Ang sarap ng pagkain, mabilis pa ang delivery!",
    "Matagal dumating at malamig na pagkain",
]

results = sentiment_pipeline(test_reviews)
print("\nLanguage coverage test:")
print(f"{'Text':<50} {'Label':<15} {'Confidence'}")
print("-" * 75)
for text, result in zip(test_reviews, results):
    print(f"'{text[:48]}' → {result['label']:<15} {round(result['score']*100, 1)}%")

In [ ]:
# Malay/Rojak stress test — demonstrates failure mode on mixed-sentiment constructions

malay_tests = [
    # Clear positives
    "Sedap gila makanan dia, memang akan order lagi!",
    "Best gila, rider pun peramah, makanan panas lagi sampai",
    "Syok ah makan sini, portion besar harga berpatutan",
    "Sedap dan cepat, 5 bintang dari aku",
    "Terima kasih, makanan sedap dan packaging cantik",
    
    # Clear negatives  
    "Makanan sejuk, lambat sampai, memang hampa gila",
    "Tak sedap langsung, rugi duit aku order sini",
    "Rider lambat, makanan dah basi bila sampai",
    "Portion kecil sangat tapi harga mahal, tak berbaloi",
    "Worst experience ever, takkan order dah lepas ni",
    
    # Neutral/mixed
    "Ok je, biasa biasa sahaja",
    "Sedap tapi portion kecil sikit",
    "Laju tapi packaging bocor",
    
    # Rojak
    "Best gila babi, will definitely order again lah!",
    "So disappointed, makanan sejuk and rider was rude",
    "Not bad la, tapi agak lambat sikit delivery dia",
    "Sedap but overpriced kot, oklah untuk sekali sekala",
    
    # SMS style short reviews
    "sdp gila",
    "xbest langsung",
    "ok je",
]

print(f"{'Text':<55} {'Label':<15} {'Confidence'}")
print("-" * 85)
results = sentiment_pipeline(malay_tests)
for text, result in zip(malay_tests, results):
    confidence = round(result['score']*100, 1)
    print(f"'{text[:53]}' -> {result['label']:<15} {confidence}%")

---

## 10. Keyword Analysis

Extracts the most frequently mentioned keywords in positive (4–5 star) and negative (1–2 star) reviews per market and language group. The output feeds the Keywords dashboard in Tableau, where the user can compare what customers consistently praise versus complain about across markets.

### Country-language scope decisions

| Market | Languages analysed | Excluded |
|---|---|---|
| Malaysia | English/Malay (Latin) + Chinese | Chinese excluded — only ~13k reviews (~2% of MY volume), too thin to be representative |
| Singapore | English | Chinese excluded — only ~2k reviews (~1.8%) |
| Philippines | English | — |
| Taiwan | Chinese | — |
| Hong Kong | Chinese + English | — |

### Pipeline per slice (country × language × sentiment)

1. **Tokenisation.** Latin text via regex (`\b[a-zA-Z]{3,}\b`); Chinese text via `jieba.cut()`.
2. **CJK filter.** Restricts Chinese slices to the Unicode range `\u4e00`–`\u9fff` to strip Latin noise that leaked through.
3. **Stopword removal.** Combines: standard English NLP stopwords; a domain stopword list (food, rider, delivery, etc. — universally present and analytically uninformative); a Malay stopword list (AI-generated, then validated against the analyst's native Malay knowledge); a Chinese stopword list of common particles. Stopword lists were refined iteratively — extract, inspect output, add noise to stopwords, rerun.
4. **Frequency ranking.** Top 30 keywords per slice via `Counter.most_common`.

### `share_of_voice` metric

Keyword count divided by slice total. This normalises for volume differences — a keyword appearing 500 times in a 5,000-review slice is more notable than the same keyword in a 500,000-review slice.

### Post-processing

Chinese characters that leaked into Latin-language slices were removed via an `is_latin()` regex check.

In [7]:
import jieba
from collections import Counter
import re

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [8]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='jieba')
import jieba

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Methodology reference — not executed in this notebook.
# Output is preserved in foodpanda_keywords.csv (loaded above as keywords_df).
# ─────────────────────────────────────────────────────────────────

# Stopwords refined iteratively over multiple extraction passes
# Consolidated here from the iterative development process

stopwords = set([
    # English function words
    'the', 'a', 'an', 'is', 'it', 'and', 'or', 'but', 'in', 'on',
    'at', 'to', 'for', 'of', 'was', 'were', 'are', 'be', 'been',
    'have', 'has', 'had', 'do', 'did', 'does', 'will', 'would',
    'could', 'should', 'may', 'might', 'this', 'that', 'they',
    'them', 'with', 'from', 'by', 'as', 'my', 'your', 'our',
    'also', 'just', 'get', 'got', 'when', 'so', 'all', 'very',
    'too', 'more', 'not', 'no', 'you', 'we', 'always', 'still',
    'even', 'only', 'than', 'then', 'here', 'there', 'what',
    'how', 'which', 'who', 'order', 'ordered', 'thi', 'hai',
    'don', 'didn', 'can', 'its', 'his', 'her', 'their', 'shall',
    'these', 'those',
    
    # Topic words (universally present, not sentiment-bearing)
    'food', 'rice', 'chicken', 'delivery', 'deliver', 'rider',
    'time', 'place', 'restaurant', 'shop', 'store', 'item',
    'items', 'taste', 'eat', 'ate', 'try', 'make', 'made',
    'come', 'came', 'back', 'well', 'give', 'given', 'use',
    'used', 'say', 'said', 'know', 'want', 'need', 'wait',
    'highly', 'overall', 'definitely', 'actually', 'every',
    'one', 'much', 'really', 'thank', 'again', 'price',
    'received', 'lot', 'day', 'first', 'last', 'next',
    'good', 'bad', 'okay', 'nice',
    
    # Generic review words
    'experience', 'recommend', 'recommended', 'worth', 'value',
    'portion', 'quantity', 'quality', 'packaging', 'packed',
    'customer', 'service', 'review', 'rating', 'star', 'stars',
    'ordered', 'ordering', 'minutes', 'hours', 'long', 'short',
    'bit', 'little', 'quite', 'rather', 'enough', 'less',
    'better', 'best', 'worst',
    'like', 'just', 'also', 'even', 'still', 'already',
    'sometimes', 'usually', 'normally', 'always', 'never',
    
    # Malay function words
    'yang', 'dan', 'di', 'ke', 'dengan', 'untuk', 'ada', 'ini',
    'itu', 'je', 'la', 'lah', 'kan', 'pun', 'dah', 'ok', 'tak',
    'nak', 'saya', 'dia', 'kami', 'mereka', 'tapi', 'juga',
    'sudah', 'akan', 'boleh', 'dari', 'bila', 'masa', 'kali',
    'sikit', 'sangat', 'memang', 'pon', 'nye', 'nya', 'dapat',
    'semua', 'makan', 'makanan', 'nasi', 'ayam', 'goreng',
    
    # Malay SMS abbreviations
    'mcm', 'dpt', 'dgn', 'utk', 'yg', 'xde', 'xbest',
    'sbb', 'ckp', 'bkn', 'tdk', 'nk', 'nnt', 'jgn', 'byk',
    'sgt', 'btw', 'dlm', 'dri', 'drp', 'sprt',
    'org', 'ade', 'tgh', 'jap', 'sat', 'lagi', 'mmg',
    
    # Malay common verbs/neutral words
    'rasa', 'bagi', 'beli', 'sampai', 'masak', 'ambil', 'letak',
    'minta', 'bayar', 'hantar', 'tunggu', 'pergi',
    'datang', 'cakap', 'bagus', 'lain', 'macam', 'sama', 'tiap',
    
    # Malay food items
    'mee', 'sup', 'kuah', 'roti', 'telur', 'daging',
    'ikan', 'sayur', 'sambal', 'pedas', 'manis', 'asin', 'lemak',
    'bihun', 'laksa', 'rendang', 'kueh',
    'teh', 'kopi', 'air', 'ais', 'buah', 'lauk', 'mihun',
    
    # English food items
    'burger', 'pizza', 'fries', 'cheese', 'sauce', 'bread',
    'pasta', 'salad', 'soup', 'meat', 'beef', 'pork', 'fish',
    'egg', 'noodle', 'spicy', 'sweet', 'salty',
    'biryani', 'curry', 'wings', 'wrap', 'bowl', 'set',
    
    # Malay function words (additional)
    'apa', 'dalam', 'kalau', 'atau', 'kena',
    'pasal', 'sebab', 'camne', 'macamne', 'camtu',
    'selalu', 'kadang', 'mesti', 'perlu', 'antara', 'setiap',
    'satu', 'dua', 'tiga', 'empat', 'lima',
    'hari', 'waktu', 'minggu', 'bulan',
    'tahun', 'pagi', 'petang', 'malam', 'sini', 'sana', 'situ',
    
    # Malay topic/neutral words
    'kedai', 'warung', 'tempat', 'lokasi', 'alamat', 'kawasan',
    'harga', 'payment', 'checkout', 'oder',
    'staff', 'pekerja', 'owner', 'peniaga', 'operator',
    
    # Positive words appearing in negative reviews due to negation
    'sedap', 'enak', 'lazat', 'lezat', 'nikmat',
    
    # More SMS abbreviations
    'ape', 'npe', 'mcmne', 'mcmna', 'sape', 'mane',
    'gne', 'bile', 'kne', 'ble',
    
    # Malay pronouns
    'aku', 'kau', 'kita', 'korang', 'awak',
    
    # Malay verbs/neutral
    'buat', 'mkn', 'minum', 'bawa',
    'pakai', 'guna', 'tanya', 'jawab', 'call', 'contact',
    
    # Malay question words
    'mana', 'mane', 'kenapa', 'knapa',
    
    # Food items remaining
    'tomyam', 'sos', 'kuih', 'nasik', 'ulam',
])

# Chinese stopwords
chinese_stopwords = set([
    '的', '了', '和', '是', '在', '我', '有', '就', '不', '也',
    '都', '但', '很', '这', '他', '她', '它', '们', '个', '上',
    '到', '说', '要', '去', '你', '会', '着', '没', '看', '好',
    '来', '把', '被', '让', '给', '从', '得', '与', '及', '或',
    '而', '由', '对', '更', '再', '又', '已', '将', '可', '还',
    '之', '其', '此', '该', '所', '以', '于', '为', '因', '如',
    '那', '啊', '吧', '呢', '哦', '嗯', '哈', '嘛', '啦', '喔',
    '嗎', '唷', '耶', '囉', '吼', '欸', '齁', '喵', '哩', '咧',
    '店', '這',
    # Chinese food/delivery topic words
    '外卖', '送餐', '骑手', '餐厅', '食物', '味道', '价格',
])

print(f"Latin stopwords: {len(stopwords)}")
print(f"Chinese stopwords: {len(chinese_stopwords)}")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Methodology reference — not executed in this notebook.
# Output is preserved in foodpanda_keywords.csv (loaded above as keywords_df).
# ─────────────────────────────────────────────────────────────────

# Country-language scope configuration
# Chinese excluded from Malaysia and Singapore slices — review volume too low to be representative

country_language_config = {
    'Malaysia': {
        'English/Malay Latin': ['en', 'ms', 'id'],
    },
    'Singapore': {
        'English': ['en'],
    },
    'Philippines': {
        'English': ['en'],
    },
    'Taiwan': {
        'Chinese': ['zh-tw', 'zh-cn'],
    },
    'Hong Kong': {
        'Chinese': ['zh-tw', 'zh-cn'],
        'English': ['en'],
    },
}

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Methodology reference — not executed in this notebook.
# Output is preserved in foodpanda_keywords.csv (loaded above as keywords_df).
# ─────────────────────────────────────────────────────────────────

def extract_keywords_by_language(country, language_label, language_codes, sentiment, top_n=30):
    """Extract top keywords for a country x language x sentiment slice."""
    
    ratings = [1, 2] if sentiment == 'negative' else [4, 5]
    
    texts = master_APAC[
        (master_APAC['country'] == country) &
        (master_APAC['language'].isin(language_codes)) &
        (master_APAC['overall'].isin(ratings))
    ]['text'].dropna()
    
    if len(texts) == 0:
        return []
    
    is_chinese = language_label == 'Chinese'
    all_words = []
    
    for text in texts:
        text_str = str(text)
        
        if is_chinese:
            # Chinese tokenisation via jieba
            words = jieba.cut(text_str)
            # Restrict to CJK Unicode range to strip Latin noise
            filtered = [w for w in words
                       if w not in chinese_stopwords
                       and len(w) >= 2
                       and any('\u4e00' <= c <= '\u9fff' for c in w)]
        else:
            # Latin tokenisation via regex
            words = re.findall(r'\b[a-zA-Z]{3,}\b', text_str.lower())
            filtered = [w for w in words if w not in stopwords]
        
        all_words.extend(filtered)
    
    results = []
    for word, count in Counter(all_words).most_common(top_n):
        results.append({
            'country': country,
            'language_group': language_label,
            'sentiment': sentiment,
            'keyword': word,
            'count': count
        })
    
    return results

In [ ]:
# Executed once during pipeline build; cached output loaded below for inspection
# Runs jieba tokenisation on the full corpus — expensive for Chinese markets

# all_results = []
# for country, language_groups in country_language_config.items():
#     print(f"\nProcessing {country}...")
#     for language_label, language_codes in language_groups.items():
#         for sentiment in ['negative', 'positive']:
#             results = extract_keywords_by_language(
#                 country, language_label,
#                 language_codes, sentiment, top_n=30
#             )
#             all_results.extend(results)
#
# keywords_df = pd.DataFrame(all_results)

In [9]:
# ─────────────────────────────────────────────────────────────────
# Methodology reference — not executed in this notebook.
# Output is preserved in foodpanda_keywords.csv (loaded above as keywords_df).
# ─────────────────────────────────────────────────────────────────

# Load cached keyword output
keywords_df = pd.read_csv('foodpanda_keywords.csv')

print(f"Keywords loaded: {len(keywords_df)} rows")
print(f"\nSlice breakdown:")
print(keywords_df.groupby(['country', 'language_group', 'sentiment'])['keyword'].count())

Keywords loaded: 480 rows

Slice breakdown:
country      language_group       sentiment
Hong Kong    Chinese              negative     60
                                  positive     60
             English              negative     30
                                  positive     30
Malaysia     English/Malay Latin  negative     30
                                  positive     30
Philippines  English              negative     30
                                  positive     30
Singapore    English              negative     30
                                  positive     30
Taiwan       Chinese              negative     60
                                  positive     60
Name: keyword, dtype: int64


In [ ]:
# ─────────────────────────────────────────────────────────────────
# Methodology reference — not executed in this notebook.
# Output is preserved in foodpanda_keywords.csv (loaded above as keywords_df).
# ─────────────────────────────────────────────────────────────────

# share_of_voice normalises for slice size — raw counts aren't comparable across markets
print("\nShare of voice sample (Malaysia, English/Malay Latin, negative):")
print(keywords_df[
    (keywords_df['country'] == 'Malaysia') &
    (keywords_df['language_group'] == 'English/Malay Latin') &
    (keywords_df['sentiment'] == 'negative')
][['keyword', 'count', 'share_of_voice']].head(15))

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Methodology reference — not executed in this notebook.
# Output is preserved in foodpanda_keywords.csv (loaded above as keywords_df).
# ─────────────────────────────────────────────────────────────────

# is_latin() removes Chinese characters that leaked into Latin slices
def is_latin(text):
    return bool(re.match(r'^[a-zA-Z\s]+$', str(text)))

# Verify no Chinese characters remain in Latin groups
latin_groups = ['English/Malay Latin', 'English']
check = keywords_df[keywords_df['language_group'].isin(latin_groups)]['keyword']
chinese_leaked = check[~check.apply(is_latin)]
print(f"Chinese characters in Latin groups: {len(chinese_leaked)}")
if len(chinese_leaked) > 0:
    print(chinese_leaked.tolist())

In [10]:
# Verification: keyword output
print(f"keywords_df shape: {keywords_df.shape}")
print(f"\nCountries covered: {keywords_df['country'].unique()}")
print(f"Language groups: {keywords_df['language_group'].unique()}")
print(f"\nTop 10 positive keywords for Malaysia (English/Malay Latin):")
my_pos = keywords_df[(keywords_df['country']=='Malaysia') & 
                      (keywords_df['language_group']=='English/Malay Latin') & 
                      (keywords_df['sentiment']=='positive')].head(10)
print(my_pos[['keyword', 'count', 'share_of_voice']])

keywords_df shape: (480, 7)

Countries covered: ['Malaysia' 'Singapore' 'Philippines' 'Taiwan' 'Hong Kong']
Language groups: ['English/Malay Latin' 'English' 'Chinese']

Top 10 positive keywords for Malaysia (English/Malay Latin):
          keyword  count  share_of_voice
30         banyak  27693            8.44
31        terbaik  26638            8.12
32         repeat  23830            7.27
33         terima  17743            5.41
34          kasih  17440            5.32
35           cuma  16165            4.93
36      delicious  14703            4.48
37  alhamdulillah  12734            3.88
38           puas  11651            3.55
39           hati  11597            3.54


> **Limitations:**
> - **Unigram analysis only.** Negation is not captured — *"tidak sedap"* ("not tasty") splits into the tokens *tidak* and *sedap*, and the keyword `sedap` then appears in negative reviews despite expressing the opposite of what it means in context. Bigram analysis would address this and is a candidate for a future iteration.
> - **Chinese keywords (Taiwan, Hong Kong) were not validated by a native Chinese speaker.** Tokenisation by `jieba` is generally reliable, but stopword sufficiency and contextual interpretation of remaining keywords would benefit from native review.
> - **Malay/Indonesian reviews from non-sentiment countries (e.g. Bangladesh, Pakistan) are excluded from keyword analysis** — scope was bounded to the five core sentiment markets.

---

## 11. Final Export

Selects 24 columns from `master_APAC` for the Tableau workbook and exports two files:

- **`foodpanda_APAC_final.csv`** (~1.3GB) — primary analytical table, one row per review, joined with restaurant metadata and engineered features.
- **`foodpanda_keywords.csv`** — secondary table powering the Keywords dashboard, one row per (country, language_group, sentiment, keyword).

Columns excluded from the export are either unused in the dashboard (e.g. `replies`, `isLiked`) or are intermediate fields from earlier processing steps.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Methodology reference — not executed in this notebook.
# Output is preserved in foodpanda_APAC_final.csv (loaded above as final_df).
# ─────────────────────────────────────────────────────────────────

# 24-column export — keeps Tableau extract size manageable while preserving all dashboard fields

tableau_columns = [
    # Identifiers
    'StoreId', 'uuid', 'country', 
    # Restaurant info
    'CompleteStoreName', 'FoodType', 'AverageRating', 
    'Reviewers', 'City', 'Location',
    # Singapore specific
    'region', 'latitude', 'longitude',
    # Review info
    'createdAt', 'text', 'overall', 'rider', 
    'restaurant_food', 'likeCount', 'isAnonymous',
    # Derived
    'rating_sentiment', 'sentiment_label',
    'year', 'month', 'year_month'
]

# Only keep columns that exist
existing_columns = [col for col in tableau_columns 
                   if col in master_APAC.columns]

tableau_df = master_APAC[existing_columns].copy()

print("Final Tableau dataset:")
print(f"Rows: {len(tableau_df):,}")
print(f"Columns: {len(existing_columns)}")
print(f"\nColumns included: {existing_columns}")
print(f"\nNull check on key columns:")
print(tableau_df[['country', 'overall', 'sentiment_label', 
                   'year', 'CompleteStoreName']].isnull().sum())

In [ ]:
# Executed once during pipeline build; cached output loaded below for inspection

# output_path = 'foodpanda_APAC_final.csv'
# print("Exporting final dataset... this may take a few minutes given the size")
# tableau_df.to_csv(output_path, index=False)
# file_size = round(os.path.getsize(output_path) / 1024 / 1024, 2)
# print(f"Export complete! Size: {file_size} MB, Rows: {len(tableau_df):,}")

In [11]:
# Verification: final export files
import os
for f in ['foodpanda_APAC_final.csv', 'foodpanda_keywords.csv']:
    if os.path.exists(f):
        size_mb = os.path.getsize(f) / (1024**2)
        print(f"{f}: {size_mb:.1f} MB")
print(f"\nfinal_df: {final_df.shape}")
print(f"keywords_df: {keywords_df.shape}")

foodpanda_APAC_final.csv: 1376.8 MB
foodpanda_keywords.csv: 0.0 MB

final_df: (4604685, 24)
keywords_df: (480, 7)


---

## Methodology Summary & Next Iteration

### Acknowledged limitations

- **Singapore region coverage** at approximately 60%; remaining 40% labelled `Unknown`.
- **Language detection** is statistical, with a known Traditional Chinese misclassification corrected; smaller errors likely remain.
- **Keyword analysis is unigram-only** — negation not captured.
- **Chinese keyword interpretation** not validated by a native speaker.

### Candidate next iterations

- Bigram keyword analysis to capture negation patterns.
- Address standardisation pipeline before OneMap geocoding to lift Singapore coverage.
- Re-attempt Malaya in a Python 3.11 environment for Malay NLP comparison.
- Native Chinese review of Taiwan / Hong Kong keyword outputs.

### Project context

- **Thailand exited Foodpanda in May 2025** — visible as a hard stop in trend data.
- **Taiwan acquisition by Grab** is pending H2 2026 — relevant context for Taiwan-specific findings.